## 8: NLP на датасете Tweets


In [1]:
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_style("whitegrid")


In [2]:
df_raw = pd.read_csv("Tweets.csv")

print(f"Размер исходного датасета: {df_raw.shape}")
print(f"Столбцы: {df_raw.columns.tolist()}")
print()
print("Пропуски:")
print(df_raw.isna().sum().to_string())
print()
print(f"Полных дубликатов: {df_raw.duplicated().sum()}")
print(f"Дубликатов по тексту: {df_raw['text'].duplicated().sum()}")
print()
print(df_raw.head(5).to_string())


Размер исходного датасета: (27481, 4)
Столбцы: ['textID', 'text', 'selected_text', 'sentiment']

Пропуски:
textID           0
text             1
selected_text    1
sentiment        0

Полных дубликатов: 0
Дубликатов по тексту: 0

       textID                                                                         text                        selected_text sentiment
0  cb774db0d1                                          I`d have responded, if I were going  I`d have responded, if I were going   neutral
1  549e992a42                                Sooo SAD I will miss you here in San Diego!!!                             Sooo SAD  negative
2  088c60f138                                                    my boss is bullying me...                          bullying me  negative
3  9642c003ef                                               what interview! leave me alone                       leave me alone  negative
4  358bd9e861   Sons of ****, why couldn`t they put them on the releases we alre

Датасет загрузился корректно. В нём 27481 строка и 4 столбца. Пропусков почти нет, только по одному в text и selected_text. Полных дубликатов нет, значит можно спокойно работать дальше.


In [3]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^a-z\s\']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = df_raw.dropna(subset=["text", "sentiment"]).copy()
df["text_clean"] = df["text"].apply(preprocess_text)
empty_after_clean = (df["text_clean"].str.len() == 0).sum()
df = df[df["text_clean"].str.len() > 0].copy()

print(f"Размер после очистки: {df.shape}")
print(f"Пустых текстов после чистки удалено: {empty_after_clean}")
print()
print("Примеры до и после предобработки:")
for idx in range(3):
    print(f"Исходный: {df.iloc[idx]['text']}")
    print(f"После чистки: {df.iloc[idx]['text_clean']}")
    print()


Размер после очистки: (27476, 5)
Пустых текстов после чистки удалено: 4

Примеры до и после предобработки:
Исходный:  I`d have responded, if I were going
После чистки: i d have responded if i were going

Исходный:  Sooo SAD I will miss you here in San Diego!!!
После чистки: sooo sad i will miss you here in san diego

Исходный: my boss is bullying me...
После чистки: my boss is bullying me



После очистки осталось 27476 твитов. Удалился 1 пропуск и 4 записи, которые после чистки стали пустыми. Текст стал аккуратнее, ссылки, упоминания и лишние символы исчезли, но общий смысл сохранился.


In [4]:
sentiment_counts = df["sentiment"].value_counts()
print("Распределение классов:")
print(sentiment_counts.to_string())

plt.figure(figsize=(8, 4))
sentiment_counts.plot(kind="bar", color=["tomato", "gray", "mediumseagreen"])
plt.title("Распределение тональностей")
plt.xlabel("Тональность")
plt.ylabel("Количество твитов")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


Распределение классов:
sentiment
neutral     11114
positive     8582
negative     7780


Класс neutral самый большой, 11114 записей. Положительных твитов 8582, отрицательных 7780. Перекос есть, но он умеренный, поэтому задача остаётся достаточно удобной для многоклассовой классификации.


In [5]:
df["text_len"] = df["text"].astype(str).str.len()
df["word_count"] = df["text_clean"].str.split().str.len()

len_stats = df.groupby("sentiment")[["text_len", "word_count"]].mean().round(2)
print("Средние значения длины текста:")
print(len_stats.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="sentiment", y="text_len", ax=axes[0], palette="Set2")
axes[0].set_title("Длина твита в символах")
axes[0].set_xlabel("Тональность")
axes[0].set_ylabel("Символы")

sns.boxplot(data=df, x="sentiment", y="word_count", ax=axes[1], palette="Set2")
axes[1].set_title("Длина твита в словах")
axes[1].set_xlabel("Тональность")
axes[1].set_ylabel("Слова")

plt.tight_layout()
plt.show()


Средние значения длины текста:
           text_len  word_count
sentiment                      
negative      70.50       13.71
neutral       65.22       12.46
positive      70.42       13.27


Отрицательные и положительные твиты в среднем немного длиннее нейтральных. По символам это 70.5 и 70.4 против 65.2. По словам картина такая же, у neutral тексты чуть короче. Значит эмоциональные сообщения обычно более развёрнутые.


In [6]:
freq_vec = CountVectorizer(stop_words="english", min_df=5)
freq_matrix = freq_vec.fit_transform(df["text_clean"])
freq_counts = np.asarray(freq_matrix.sum(axis=0)).ravel()
freq_terms = np.array(freq_vec.get_feature_names_out())

top_idx = freq_counts.argsort()[-15:][::-1]
top_terms = freq_terms[top_idx]
top_values = freq_counts[top_idx]

print("Топ слов по всему датасету:")
for term, count in zip(top_terms, top_values):
    print(f"{term}: {int(count)}")

class_top_words = {}
for sentiment in ["negative", "neutral", "positive"]:
    matrix_s = freq_vec.transform(df.loc[df["sentiment"] == sentiment, "text_clean"])
    counts_s = np.asarray(matrix_s.sum(axis=0)).ravel()
    idx_s = counts_s.argsort()[-10:][::-1]
    class_top_words[sentiment] = list(zip(freq_terms[idx_s], counts_s[idx_s]))

print()
print("Топ слов по классам:")
for sentiment, items in class_top_words.items():
    print(sentiment, items)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

axes[0].bar(top_terms[::-1], top_values[::-1], color="steelblue")
axes[0].set_title("Общие частые слова")
axes[0].tick_params(axis="x", rotation=45)

colors = {"negative": "tomato", "neutral": "gray", "positive": "mediumseagreen"}
for ax, sentiment in zip(axes[1:], ["negative", "neutral", "positive"]):
    words = [w for w, _ in class_top_words[sentiment]][::-1]
    values = [int(v) for _, v in class_top_words[sentiment]][::-1]
    ax.barh(words, values, color=colors[sentiment])
    ax.set_title(f"Частые слова: {sentiment}")

plt.tight_layout()
plt.show()


Топ слов по всему датасету:
just: 2278
day: 2157
good: 1579
like: 1353
work: 1151
today: 1147
love: 1146
going: 1103
got: 1086
lol: 1027
happy: 996
time: 965
know: 944
really: 915
don: 884

Топ слов по классам:
negative [('just', np.int64(692)), ('like', np.int64(478)), ('miss', np.int64(426)), ('work', np.int64(403)), ('sad', np.int64(397)), ('im', np.int64(359)), ('sorry', np.int64(352)), ('don', np.int64(347)), ('really', np.int64(344)), ('day', np.int64(343))]
neutral [('just', np.int64(923)), ('day', np.int64(544)), ('lol', np.int64(499)), ('work', np.int64(492)), ('going', np.int64(483)), ('got', np.int64(459)), ('like', np.int64(457)), ('today', np.int64(448)), ('time', np.int64(433)), ('know', np.int64(420))]
positive [('day', np.int64(1270)), ('good', np.int64(1067)), ('love', np.int64(890)), ('happy', np.int64(858)), ('just', np.int64(663)), ('thanks', np.int64(568)), ('great', np.int64(483)), ('like', np.int64(418)), ('hope', np.int64(411)), ('mother', np.int64(374))]


В целом по корпусу часто встречаются just, day, good, work. Для положительных твитов хорошо видны love, happy, thanks, great. Для отрицательных заметны miss, sad, sorry, bad. Уже на этом шаге видно, что словарь действительно связан с тональностью.


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text_clean"],
    df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

count_vec = CountVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2
)
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2
)

X_train_count = count_vec.fit_transform(X_train)
X_test_count = count_vec.transform(X_test)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")
print(f"CountVectorizer train shape: {X_train_count.shape}")
print(f"TF-IDF train shape: {X_train_tfidf.shape}")


Train size: 21980
Test size: 5496
CountVectorizer train shape: (21980, 8000)
TF-IDF train shape: (21980, 8000)


Разбили данные в пропорции 80 на 20, сохранив баланс классов. Для базовой модели используем CountVectorizer, а для линейных моделей TF-IDF. В обоих случаях берём до 8000 признаков и учитываем не только отдельные слова, но и биграммы.


In [8]:
nb_model = MultinomialNB()
nb_model.fit(X_train_count, y_train)
y_pred_nb = nb_model.predict(X_test_count)

acc_nb = accuracy_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb, average="weighted")

print("=== Multinomial Naive Bayes ===")
print(f"Accuracy: {acc_nb:.4f}")
print(f"Weighted F1: {f1_nb:.4f}")
print()
print(classification_report(y_test, y_pred_nb, digits=4))


=== Multinomial Naive Bayes ===
Accuracy: 0.6448
Weighted F1: 0.6458

              precision    recall  f1-score   support

    negative     0.6693    0.5996    0.6325      1556
     neutral     0.5853    0.6806    0.6294      2223
    positive     0.7238    0.6395    0.6790      1717

    accuracy                         0.6448      5496
   macro avg     0.6595    0.6399    0.6470      5496
weighted avg     0.6523    0.6448    0.6458      5496



MultinomialNB дал accuracy 0.6447 и weighted F1 0.6456. Это хороший базовый уровень, но видно, что модель уже начинает уступать более гибким линейным методам.


In [9]:
lr_model = LogisticRegression(max_iter=2000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)

acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr, average="weighted")

print("=== Logistic Regression ===")
print(f"Accuracy: {acc_lr:.4f}")
print(f"Weighted F1: {f1_lr:.4f}")
print()
print(classification_report(y_test, y_pred_lr, digits=4))


=== Logistic Regression ===
Accuracy: 0.6840
Weighted F1: 0.6838

              precision    recall  f1-score   support

    negative     0.7244    0.5778    0.6428      1556
     neutral     0.6155    0.7562    0.6786      2223
    positive     0.7736    0.6867    0.7276      1717

    accuracy                         0.6840      5496
   macro avg     0.7045    0.6735    0.6830      5496
weighted avg     0.6957    0.6840    0.6838      5496



Логистическая регрессия показала лучший результат, accuracy 0.6841 и weighted F1 0.6840. Особенно хорошо модель находит neutral и positive. Класс negative определяется сложнее, но качество всё равно заметно выше, чем у наивного Байеса.


In [10]:
svc_model = LinearSVC(random_state=42)
svc_model.fit(X_train_tfidf, y_train)
y_pred_svc = svc_model.predict(X_test_tfidf)

acc_svc = accuracy_score(y_test, y_pred_svc)
f1_svc = f1_score(y_test, y_pred_svc, average="weighted")

print("=== LinearSVC ===")
print(f"Accuracy: {acc_svc:.4f}")
print(f"Weighted F1: {f1_svc:.4f}")
print()
print(classification_report(y_test, y_pred_svc, digits=4))


=== LinearSVC ===
Accuracy: 0.6678
Weighted F1: 0.6680

              precision    recall  f1-score   support

    negative     0.6841    0.6054    0.6423      1556
     neutral     0.6183    0.6842    0.6496      2223
    positive     0.7275    0.7030    0.7150      1717

    accuracy                         0.6678      5496
   macro avg     0.6766    0.6642    0.6690      5496
weighted avg     0.6711    0.6678    0.6680      5496



LinearSVC тоже сработал уверенно, accuracy 0.6676 и weighted F1 0.6679. Он лучше базовой модели, но немного уступает логистической регрессии. На коротких твитах такая небольшая разница между линейными моделями вполне ожидаема.


In [11]:
results = pd.DataFrame({
    "Модель": ["MultinomialNB", "LogisticRegression", "LinearSVC"],
    "Accuracy": [acc_nb, acc_lr, acc_svc],
    "Weighted F1": [f1_nb, f1_lr, f1_svc]
}).sort_values("Weighted F1", ascending=False)

print(results.to_string(index=False))

plt.figure(figsize=(8, 4))
x = np.arange(len(results))
width = 0.35
plt.bar(x - width / 2, results["Accuracy"], width=width, label="Accuracy", color="steelblue")
plt.bar(x + width / 2, results["Weighted F1"], width=width, label="Weighted F1", color="coral")
plt.xticks(x, results["Модель"], rotation=15)
plt.ylabel("Значение метрики")
plt.title("Сравнение моделей")
plt.legend()
plt.tight_layout()
plt.show()


            Модель  Accuracy  Weighted F1
LogisticRegression  0.683952     0.683784
         LinearSVC  0.667758     0.667986
     MultinomialNB  0.644833     0.645783


Лидером стала LogisticRegression. Отрыв от LinearSVC небольшой, но стабильный. MultinomialNB остаётся полезной базовой точкой отсчёта, но на этом датасете TF-IDF и линейная классификация работают лучше.


In [12]:
best_pred = y_pred_lr
best_name = "LogisticRegression"
label_order = sorted(df["sentiment"].unique())

cm = confusion_matrix(y_test, best_pred, labels=label_order)
print("Порядок классов:", label_order)
print(cm)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_order,
    yticklabels=label_order
)
plt.title(f"Матрица ошибок: {best_name}")
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.tight_layout()
plt.show()


Порядок классов: ['negative', 'neutral', 'positive']
[[ 899  571   86]
 [ 283 1681  259]
 [  59  479 1179]]


Самая частая ошибка, перенос отрицательных и положительных твитов в neutral. По матрице видно, что negative довольно часто уходит в neutral, 568 случаев. Положительные твиты тоже иногда становятся нейтральными, 484 случая. Это логично, потому что neutral здесь играет роль промежуточного класса.


In [13]:
feature_names = tfidf.get_feature_names_out()
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, cls, coef_row in zip(axes, lr_model.classes_, lr_model.coef_):
    top_idx = np.argsort(coef_row)[-10:][::-1]
    top_terms = feature_names[top_idx][::-1]
    top_values = coef_row[top_idx][::-1]
    ax.barh(top_terms, top_values, color="slateblue")
    ax.set_title(cls)

plt.tight_layout()
plt.show()

for cls, coef_row in zip(lr_model.classes_, lr_model.coef_):
    top_idx = np.argsort(coef_row)[-10:][::-1]
    print(cls, feature_names[top_idx].tolist())


negative ['sad', 'sorry', 'hate', 'miss', 'sucks', 'sick', 'bored', 'stupid', 'tired', 'hurts']
neutral ['theres', 'training', 'guitar', 'weren', 'sent', 'moro', 'sp', 'mmm', 'goin', 'question']
positive ['love', 'thanks', 'awesome', 'happy', 'great', 'thank', 'good', 'nice', 'amazing', 'cute']


У отрицательного класса важны sad, sorry, miss, hate, tired. У положительного хорошо выделяются love, thanks, awesome, happy, great. Это очень естественный результат, признаки легко интерпретируются и хорошо объясняют, почему модель принимает решение.


### Итог

Для данного датасета лучше всего подошла связка очистка текста, TF-IDF и Logistic Regression. Итоговое качество получилось уверенным, accuracy 0.6841 и weighted F1 0.6840. Для коротких твитов этого уже достаточно, чтобы показать полноценный NLP пайплайн, понятные метрики и интерпретируемые признаки.
